# Inference pipeline

### Setup

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))


In [ ]:
import gc
import json
import os

from catboost import CatBoostRegressor
from dotenv import load_dotenv
import lightgbm as lgb
import mlflow
import numpy as np
import polars as pl
from scipy.special import ndtri
import torch
from torch import nn
import xgboost as xgb

DATA = ROOT / "data"
MODELS = ROOT / "models"
ARCHIVE = ROOT / "archive"
REFERENCE = ARCHIVE / "reference"
SUBS = ROOT / "submissions"
SETUP = DATA / "setup"
FEAT = DATA / "features"
SEQ = DATA / "seq"
MEMBER_DIR = MODELS / "members"
COMBINER_DIR = MODELS / "combiner"
SUBS.mkdir(parents=True, exist_ok=True)

VPS_IP = "2.26.27.187"
EXPERIMENT = "ecup-prod"
load_dotenv(ROOT / ".env")
mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", f"https://{VPS_IP}"))
mlflow.set_experiment(EXPERIMENT)


### Artifacts

In [ ]:
CONFIG = json.loads((SETUP / "config.json").read_text())
SUBMIT = CONFIG["submit_anchor"]
ANCHORS = CONFIG["train_anchors"]
N_USERS = CONFIG["n_users"]
USER_IDS = np.load(SETUP / "user_ids.npy")
COHORT = np.load(SETUP / "cohort.npy")
SUBMIT_MASK = COHORT[-1]

REQUIRED = [
    FEAT / "x" / f"X_{SUBMIT}.npy",
    FEAT / "e" / f"E_{SUBMIT}.npy",
    FEAT / "f" / f"F_{SUBMIT}.npy",
    FEAT / "rg" / f"RG_{SUBMIT}.npy",
    SEQ / "daily9_f16.npy",
    SEQ / "daily10_f16.npy",
    SEQ / "calendar.npy",
    COMBINER_DIR / "members.json",
    COMBINER_DIR / "nnls_kept.json",
    COMBINER_DIR / "nnls_all.json",
    COMBINER_DIR / "v46_lgbm_residual.txt",
    COMBINER_DIR / "v47_xgb_residual.ubj",
]
missing = [str(p.relative_to(ROOT)) for p in REQUIRED if not p.exists()]


### Context

In [ ]:
NAMES_X = json.loads((FEAT / "names_x.json").read_text())
NAMES_E_FULL = json.loads((FEAT / "names_e.json").read_text())
NAMES_F = json.loads((FEAT / "names_f.json").read_text())
KEEP_IDX = np.load(FEAT / "keep_idx.npy")
BASE_NAMES = [NAMES_X[j] for j in KEEP_IDX] + NAMES_E_FULL
FEATURE_NAMES = BASE_NAMES + NAMES_F
N_BASE = len(BASE_NAMES)
CAP_COLS = [
    "days_since_last_order",
    "days_since_last_cart",
    "tenure_days",
    "days_since_first_order",
]
CAP_AT = 180.0
CAP_IDX = np.array([BASE_NAMES.index(c) for c in CAP_COLS])


X445 = np.asarray(np.load(FEAT / "x" / f"X_{SUBMIT}.npy", mmap_mode="r"))[:, KEEP_IDX]
X445 = np.concatenate(
    [X445, np.asarray(np.load(FEAT / "e" / f"E_{SUBMIT}.npy", mmap_mode="r"))], axis=1
)
X445[:, CAP_IDX] = np.minimum(X445[:, CAP_IDX], CAP_AT)
X445 = np.concatenate(
    [X445, np.asarray(np.load(FEAT / "f" / f"F_{SUBMIT}.npy", mmap_mode="r"))], axis=1
)
RG445 = np.asarray(np.load(FEAT / "rg" / f"RG_{SUBMIT}.npy", mmap_mode="r"))


### Members

In [ ]:
SEED_POOL = [
    42,
    7,
    2024,
    555,
    31337,
    101,
    202,
    303,
    909,
    1234,
    5678,
    4242,
    777,
    1111,
    2222,
    3333,
    8080,
    6060,
    4040,
    2020,
]
RG_KINDS = ("mlp", "mlpce", "tabm", "mlpord")

SEQ_ARCH = {
    "v3_seq": {
        "patch": 8,
        "n_patch": 22,
        "dim": 192,
        "layers": 6,
        "heads": 6,
        "ff": 2,
        "epochs": 8,
        "valid": False,
        "side": "v3",
        "pool13": False,
        "side_hidden": 256,
        "head_hidden": 256,
        "final_norm": True,
        "mix_anchors": True,
    },
    "seq_b": {
        "patch": 7,
        "n_patch": 52,
        "dim": 256,
        "layers": 6,
        "heads": 8,
        "ff": 3,
        "epochs": 14,
        "valid": True,
        "side": "all",
        "pool13": True,
        "side_hidden": 512,
        "head_hidden": 512,
        "final_norm": True,
        "mix_anchors": True,
    },
}
SEQ2_ARCH = {
    "patch": 7,
    "n_patch": 52,
    "dim": 256,
    "layers": 6,
    "heads": 8,
    "ff": 3,
    "side_hidden": 512,
    "head_hidden": 512,
    "final_norm": True,
    "mix_anchors": True,
}
V3_SIDE_COLS = [
    "days_since_last_event",
    "days_since_last_order",
    "days_since_last_cart",
    "tenure_days",
    "active_rate_30d",
    "active_rate_90d",
    "order_day_rate_30d",
    "loggmv_per_order_365d",
    "ewm_loggmv_60",
    "ewm_has_order_60",
    "gmv_trend_7_30",
    "gmv_trend_30_90",
]

MEMBERS = {
    "catf_a": {"kind": "cat", "v4": True, "iters": 1000, "lr": 0.03, "depth": 8, "l2": 30.0},
    "catf_b": {"kind": "cat", "v4": True, "iters": 1500, "lr": 0.03, "depth": 6, "l2": 6.0},
    "catf_d": {"kind": "cat", "v4": False, "iters": 1500, "lr": 0.03, "depth": 8, "l2": 100.0},
    "catf_f": {
        "kind": "cat",
        "v4": True,
        "iters": 1200,
        "lr": 0.03,
        "depth": 10,
        "l2": 30.0,
        "extra": {"grow_policy": "Lossguide", "max_leaves": 192, "min_data_in_leaf": 128},
    },
    "cat_a": {"kind": "cat", "v4": True, "iters": 3000, "lr": 0.03, "depth": 8},
    "cat_b": {
        "kind": "cat",
        "v4": True,
        "iters": 5000,
        "lr": 0.02,
        "depth": 10,
        "l2": 12.0,
        "border": 254,
        "sub": 0.7,
    },
    "xgbf_a": {
        "kind": "xgb",
        "v4": True,
        "rounds": 150,
        "lr": 0.03,
        "depth": 9,
        "colsample": 0.5,
        "mcw": 32.0,
        "n_oof": 3,
        "n_sub": 8,
    },
    "xgbf_b": {
        "kind": "xgb",
        "v4": False,
        "rounds": 250,
        "lr": 0.02,
        "depth": 8,
        "colsample": 0.7,
        "mcw": 100.0,
        "n_oof": 3,
        "n_sub": 8,
    },
    "xgb_a": {
        "kind": "xgb",
        "v4": True,
        "rounds": 2500,
        "lr": 0.03,
        "depth": 9,
        "colsample": 0.5,
        "mcw": 32.0,
    },
    "lgbf_a": {
        "kind": "lgb",
        "v4": False,
        "rounds": 150,
        "lr": 0.03,
        "leaves": 255,
        "n_oof": 2,
        "n_sub": 6,
    },
    "lgbf_b": {
        "kind": "lgb",
        "v4": True,
        "rounds": 250,
        "lr": 0.02,
        "leaves": 127,
        "n_oof": 2,
        "n_sub": 6,
    },
    "lgb_a": {"kind": "lgb", "v4": False, "rounds": 2500, "lr": 0.03, "leaves": 255},
    "mlpf_a": {
        "kind": "mlp",
        "v4": True,
        "epochs": 4,
        "width": 2560,
        "drop": 0.35,
        "lr": 2e-3,
        "n_oof": 6,
        "n_sub": 16,
    },
    "mlpf_b": {
        "kind": "mlp",
        "v4": False,
        "epochs": 3,
        "width": 1536,
        "drop": 0.2,
        "lr": 3e-3,
        "n_oof": 6,
        "n_sub": 16,
    },
    "mlpf_c": {"kind": "mlpce", "v4": True, "epochs": 3, "width": 1536, "n_oof": 6, "n_sub": 16},
    "mlpf_d": {
        "kind": "mlp",
        "v4": True,
        "epochs": 3,
        "width": 1024,
        "drop": 0.35,
        "lr": 3e-3,
        "n_oof": 6,
        "n_sub": 16,
    },
    "mlpf_o": {"kind": "mlpord", "v4": True, "epochs": 3, "width": 1536, "n_oof": 6, "n_sub": 16},
    "tabm_a": {
        "kind": "tabm",
        "v4": True,
        "epochs": 8,
        "width": 512,
        "blocks": 3,
        "k": 32,
        "drop": 0.1,
        "lr": 2e-3,
        "ple": False,
        "n_oof": 3,
        "n_sub": 6,
    },
    "tabm_b": {
        "kind": "tabm",
        "v4": False,
        "epochs": 4,
        "width": 512,
        "blocks": 3,
        "k": 32,
        "drop": 0.1,
        "lr": 2e-3,
        "ple": True,
        "n_oof": 2,
        "n_sub": 4,
    },
    "v3_cat": {
        "kind": "cat",
        "v4": False,
        "iters": 3000,
        "lr": 0.03,
        "depth": 8,
        "l2": 6.0,
        "n_oof": 1,
        "n_sub": 3,
    },
    "v3_cat2": {
        "kind": "cat",
        "v4": False,
        "iters": 5000,
        "lr": 0.02,
        "depth": 10,
        "l2": 12.0,
        "border": 254,
        "sub": 0.7,
        "rs": 2.0,
        "n_oof": 1,
        "n_sub": 3,
    },
    "v3_mlp": {
        "kind": "mlpv3",
        "v4": False,
        "epochs": 20,
        "width": 1024,
        "drop": 0.15,
        "lr": 3e-3,
        "n_oof": 1,
        "n_sub": 3,
    },
    "v3_seq": {"kind": "seq", "v4": False, "n_oof": 1, "n_sub": 2},
    "seq_b": {"kind": "seq", "v4": False, "n_oof": 1, "n_sub": 3},
    "seq_c": {
        "kind": "seq2",
        "v4": False,
        "epochs": 3,
        "lr": 1e-3,
        "head_lr_mult": 1.0,
        "n_oof": 1,
        "n_sub": 3,
    },
}
for m in ("catf_d", "catf_b", "xgbf_a", "catf_f", "xgbf_b", "catf_a", "lgbf_b", "lgbf_a", "cat_a"):
    MEMBERS[f"{m}_sp"] = {**MEMBERS[m], "min_gap": 28}
for cfg in MEMBERS.values():
    cfg.setdefault("min_gap", 0)
    cfg.setdefault("n_oof", 2)
    cfg.setdefault("n_sub", 3)

MEMBER_ORDER = sorted(MEMBERS)
SAVED = json.loads((COMBINER_DIR / "members.json").read_text())


### Torch modules

In [ ]:
TABM_K, PLE_BINS, PLE_DIM = 32, 8, 8
HL_LO, HL_HI, HL_BINS = -2.8, 9.2, 48
HL_EDGES = np.linspace(HL_LO, HL_HI, HL_BINS + 1)
HL_CENTERS = 0.5 * (HL_EDGES[:-1] + HL_EDGES[1:])
HL_SIGMA = 0.75 * (HL_EDGES[1] - HL_EDGES[0])
ORD_LO, ORD_HI, ORD_BINS = -2.8, 9.2, 32
ORD_EDGES = np.linspace(ORD_LO, ORD_HI, ORD_BINS + 1)
ORD_CENTERS = 0.5 * (ORD_EDGES[:-1] + ORD_EDGES[1:])


class LinearBE(nn.Module):
    def __init__(self, d_in, d_out, k, first):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(d_in, d_out))
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)
        sign = torch.randint(0, 2, (k, d_in), dtype=torch.float32) * 2.0 - 1.0
        self.r = nn.Parameter(sign if first else torch.ones(k, d_in))
        self.s = nn.Parameter(torch.ones(k, d_out))
        self.bias = nn.Parameter(torch.zeros(k, d_out))

    def forward(self, x):
        return ((x * self.r) @ self.weight) * self.s + self.bias


class PLE(nn.Module):
    def __init__(self, edges):
        super().__init__()
        d, n_edge = edges.shape
        t_bins = n_edge - 1
        self.register_buffer("lo", edges[:, :-1].contiguous())
        self.register_buffer("width", (edges[:, 1:] - edges[:, :-1]).clamp_min(1e-6))
        self.weight = nn.Parameter(torch.randn(d, t_bins, PLE_DIM) * (1.0 / t_bins**0.5))
        self.bias = nn.Parameter(torch.zeros(d, PLE_DIM))
        self.out_dim = d * PLE_DIM

    def forward(self, x):
        t = ((x[..., None] - self.lo) / self.width).clamp(0.0, 1.0)
        return (torch.einsum("bdt,dte->bde", t, self.weight) + self.bias).flatten(1)


class TabM(nn.Module):
    def __init__(self, d_in, k, width, blocks, drop, n_out, edges=None):
        super().__init__()
        self.k = k
        self.emb = PLE(edges) if edges is not None else None
        d = self.emb.out_dim if self.emb is not None else d_in
        self.layers = nn.ModuleList([
            LinearBE(d if i == 0 else width, width, k, i == 0) for i in range(blocks)
        ])
        self.drop = nn.Dropout(drop)
        self.head = LinearBE(width, n_out, k, False)

    def forward(self, x):
        if self.emb is not None:
            x = self.emb(x)
        h = x[:, None].expand(-1, self.k, -1)
        for lin in self.layers:
            h = self.drop(nn.functional.gelu(lin(h)))
        return self.head(h)


class Seq(nn.Module):
    def __init__(self, arch, n_side, n_ch):
        super().__init__()
        dim = arch["dim"]
        self.valid = bool(arch["valid"])
        self.pool13 = bool(arch["pool13"])
        n_extra = 2 if self.valid else 1
        n_pool = 5 if self.pool13 else 4
        self.proj = nn.Linear(2 * n_ch + n_extra, dim)
        self.pos = nn.Parameter(torch.zeros(1, arch["n_patch"], dim))
        layer = nn.TransformerEncoderLayer(
            dim,
            arch["heads"],
            dim_feedforward=arch["ff"] * dim,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.enc = nn.TransformerEncoder(
            layer, arch["layers"], norm=nn.LayerNorm(dim) if arch["final_norm"] else None
        )
        self.attn = nn.Linear(dim, 1)
        side_layers = [nn.Linear(n_side, arch["side_hidden"]), nn.GELU()]
        if self.valid:
            side_layers.append(nn.Dropout(0.15))
        side_layers.append(nn.Linear(arch["side_hidden"], dim))
        self.side = nn.Sequential(*side_layers)
        self.head = nn.Sequential(
            nn.Linear(n_pool * dim, arch["head_hidden"]),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(arch["head_hidden"], 2),
        )

    def forward(self, seq, side):
        h = self.enc(self.proj(seq) + self.pos)
        w = torch.softmax(self.attn(h), dim=1)
        pooled = [h[:, -1], h[:, -4:].mean(1)]
        if self.pool13:
            pooled.append(h[:, -13:].mean(1))
        pooled += [(h * w).sum(1), self.side(side)]
        o = self.head(torch.cat(pooled, dim=1))
        return o[:, 0], o[:, 1]


class Encoder(nn.Module):
    def __init__(self, arch, n_feat):
        super().__init__()
        dim = arch["dim"]
        self.proj = nn.Linear(n_feat, dim)
        self.pos = nn.Parameter(torch.zeros(1, arch["n_patch"], dim))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, dim))
        layer = nn.TransformerEncoderLayer(
            dim,
            arch["heads"],
            dim_feedforward=arch["ff"] * dim,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.enc = nn.TransformerEncoder(
            layer, arch["layers"], norm=nn.LayerNorm(dim) if arch["final_norm"] else None
        )

    def forward(self, seq, mask=None):
        h = self.proj(seq)
        if mask is not None:
            h = torch.where(mask.unsqueeze(-1), self.mask_token.expand_as(h), h)
        return self.enc(h + self.pos)


class SeqNet(nn.Module):
    def __init__(self, arch, n_feat, n_side):
        super().__init__()
        dim = arch["dim"]
        self.encoder = Encoder(arch, n_feat)
        self.attn = nn.Linear(dim, 1)
        self.side = nn.Sequential(
            nn.Linear(n_side, arch["side_hidden"]),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(arch["side_hidden"], dim),
        )
        self.head = nn.Sequential(
            nn.Linear(5 * dim, arch["head_hidden"]),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(arch["head_hidden"], 2),
        )

    def forward(self, seq, side):
        h = self.encoder(seq)
        w = torch.softmax(self.attn(h), dim=1)
        pooled = [h[:, -1], h[:, -4:].mean(1), h[:, -13:].mean(1), (h * w).sum(1), self.side(side)]
        o = self.head(torch.cat(pooled, dim=1))
        return o[:, 0], o[:, 1]


### Member predictions

In [ ]:
MODEL_EXT = {
    "cat": "cbm",
    "xgb": "ubj",
    "lgb": "txt",
    "mlp": "pt",
    "mlpce": "pt",
    "mlpord": "pt",
    "mlpv3": "pt",
    "tabm": "pt",
    "seq": "pt",
    "seq2": "pt",
}
SIDE_IDX = {"all": np.arange(N_BASE)}
V3_RANK_COLS = [n for n in BASE_NAMES if n.startswith("rank_")]
SIDE_IDX["v3"] = np.array([BASE_NAMES.index(c) for c in V3_RANK_COLS + V3_SIDE_COLS])
N_CAL = 7

DAILY = {}
for tag in ("daily9", "daily10"):
    DAILY[tag] = json.loads((SEQ / f"{tag}_meta.json").read_text())
CAL_NP = np.load(SEQ / "calendar.npy")
ANCHOR_T_SUBMIT = int(
    (np.datetime64(SUBMIT) - np.datetime64(CONFIG["date_min"])) / np.timedelta64(1, "D")
)


In [ ]:
MEMBER_Z = {}
for name in MEMBER_ORDER:
    cfg = MEMBERS[name]
    kind = cfg["kind"]
    ext = MODEL_EXT[kind]
    seeds = SEED_POOL[: cfg["n_sub"]]
    preds = []
    if kind in ("seq", "seq2"):
        is_seq2 = kind == "seq2"
        arch = SEQ2_ARCH if is_seq2 else SEQ_ARCH[name]
        tag = "daily10" if is_seq2 else "daily9"
        n_ch = len(DAILY[tag]["channels"])
        side_idx = SIDE_IDX["all"] if is_seq2 else SIDE_IDX[str(arch["side"])]
        n_side = len(side_idx)
        n_feat = 2 * n_ch + 1 + 1 + N_CAL
        patch = int(arch["patch"])
        n_patch = int(arch["n_patch"])
        context_len = patch * n_patch
        daily = torch.from_numpy(np.load(SEQ / f"{tag}_f16.npy")).to("cuda")
        cal = torch.from_numpy(CAL_NP).to("cuda")
        vside = torch.from_numpy(np.ascontiguousarray(RG445[:, side_idx])).to("cuda")
        for seed in seeds:
            net = (SeqNet(arch, n_feat, n_side) if is_seq2 else Seq(arch, n_side, n_ch)).to("cuda")
            net.load_state_dict(
                torch.load(MEMBER_DIR / name / f"sub_{seed}.pt", map_location="cuda")
            )
            net.eval()
            out_chunks = []
            with torch.no_grad():
                for k in range(0, N_USERS, 4096):
                    u = torch.arange(k, min(k + 4096, N_USERS), device="cuda")
                    lo_d = ANCHOR_T_SUBMIT - context_len + 1
                    if lo_d >= 0:
                        win = daily[u, lo_d : ANCHOR_T_SUBMIT + 1].float()
                        valid = torch.ones(len(u), n_patch, 1, device="cuda")
                        cwin = cal[lo_d : ANCHOR_T_SUBMIT + 1]
                    else:
                        pad = -lo_d
                        win = torch.zeros(len(u), context_len, n_ch, device="cuda")
                        win[:, pad:] = daily[u, 0 : ANCHOR_T_SUBMIT + 1].float()
                        v = torch.ones(context_len, device="cuda")
                        v[:pad] = 0.0
                        valid = (
                            v.view(n_patch, patch).amax(dim=1).view(1, -1, 1).expand(len(u), -1, -1)
                        )
                        if is_seq2:
                            cwin = torch.zeros(context_len, cal.shape[1], device="cuda")
                            cwin[pad:] = cal[0 : ANCHOR_T_SUBMIT + 1]
                    win = win.view(len(u), n_patch, patch, n_ch)
                    if is_seq2:
                        cnt = (win[:, :, :, -1] > 0).float().sum(dim=2, keepdim=True) / patch
                        content = torch.cat([win.sum(dim=2), win.amax(dim=2), cnt], dim=2)
                        cpatch = (
                            cwin
                            .view(n_patch, patch, -1)
                            .mean(dim=1)
                            .unsqueeze(0)
                            .expand(len(u), -1, -1)
                        )
                        seq = torch.cat([content, valid, cpatch], dim=2)
                    else:
                        cnt = (win[:, :, :, -1] > 0).float().sum(dim=2, keepdim=True)
                        parts = [win.sum(dim=2), win.amax(dim=2), cnt]
                        if bool(arch["valid"]):
                            parts.append(valid)
                        seq = torch.cat(parts, dim=2)
                    with torch.autocast("cuda", dtype=torch.bfloat16):
                        mu_out, _ = net(seq, vside[k : k + 4096].float())
                    out_chunks.append(mu_out.float().cpu().numpy())
            preds.append(np.concatenate(out_chunks).astype(np.float64))
            del net
            torch.cuda.empty_cache()
        del daily, cal, vside
        torch.cuda.empty_cache()
    else:
        nc = 445 if cfg["v4"] else N_BASE
        if kind == "mlpv3":
            src = X445[:, :nc]
            xv = np.empty(src.shape, dtype=np.float32)
            nrow = src.shape[0]
            for j in range(src.shape[1]):
                v = np.asarray(src[:, j], dtype=np.float64)
                order = np.argsort(v, kind="mergesort")
                sv = v[order]
                new_grp = np.empty(nrow, dtype=bool)
                new_grp[0] = True
                np.not_equal(sv[1:], sv[:-1], out=new_grp[1:])
                grp = np.cumsum(new_grp) - 1
                counts = np.bincount(grp)
                starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
                ranked = np.empty(nrow, dtype=np.float64)
                ranked[order] = starts[grp] + (counts[grp] - 1) / 2.0
                r = (ranked / nrow + 0.5 / nrow).clip(1e-6, 1.0 - 1e-6)
                xv[:, j] = ndtri(r).astype(np.float32)
        elif kind in RG_KINDS:
            xv = RG445[:, :nc]
        else:
            xv = X445[:, :nc]
        for seed in seeds:
            path = MEMBER_DIR / name / f"sub_{seed}.{ext}"
            if kind == "cat":
                booster = CatBoostRegressor()
                booster.load_model(str(path))
                preds.append(np.asarray(booster.predict(xv), dtype=np.float64))
            elif kind == "xgb":
                booster = xgb.Booster()
                booster.load_model(str(path))
                preds.append(np.asarray(booster.inplace_predict(xv), dtype=np.float64))
            elif kind == "lgb":
                booster = lgb.Booster(model_file=str(path))
                preds.append(np.asarray(booster.predict(xv), dtype=np.float64))
            else:
                d_in = xv.shape[1]
                if kind == "tabm":
                    edges = torch.zeros(d_in, PLE_BINS + 1, device="cuda") if cfg["ple"] else None
                    model = TabM(
                        d_in, cfg["k"], cfg["width"], cfg["blocks"], cfg["drop"], 2, edges
                    ).to("cuda")
                    chunk = 8192
                else:
                    width = cfg["width"]
                    drop = cfg.get("drop", 0.2)
                    n_out = {"mlp": 3, "mlpce": HL_BINS, "mlpord": ORD_BINS - 1, "mlpv3": 2}[kind]
                    model = nn.ModuleDict({
                        "body": nn.Sequential(
                            nn.Linear(d_in, width),
                            nn.BatchNorm1d(width),
                            nn.GELU(),
                            nn.Dropout(drop),
                            nn.Linear(width, width // 2),
                            nn.BatchNorm1d(width // 2),
                            nn.GELU(),
                            nn.Dropout(drop),
                            nn.Linear(width // 2, 256),
                            nn.BatchNorm1d(256),
                            nn.GELU(),
                        ),
                        "skip": nn.Linear(d_in, 256),
                        "head": nn.Linear(256, n_out),
                    }).to("cuda")
                    chunk = 16384
                model.load_state_dict(torch.load(path, map_location="cuda"))
                model.eval()
                centers_t = torch.tensor(HL_CENTERS, dtype=torch.float32, device="cuda")
                ord_centers = torch.tensor(ORD_CENTERS, dtype=torch.float32, device="cuda")
                out_chunks = []
                with torch.no_grad():
                    for kk in range(0, len(xv), chunk):
                        xb = (
                            torch
                            .from_numpy(np.ascontiguousarray(xv[kk : kk + chunk]))
                            .to("cuda")
                            .float()
                        )
                        with torch.autocast("cuda", dtype=torch.bfloat16):
                            if kind == "tabm":
                                o = model(xb).float()
                            else:
                                h = model["body"](xb) + model["skip"](xb)
                                o = model["head"](h).float()
                        if kind == "tabm":
                            out_chunks.append(o[:, :, 0].mean(dim=1).cpu().numpy())
                        elif kind == "mlpce":
                            out_chunks.append((torch.softmax(o, dim=1) @ centers_t).cpu().numpy())
                        elif kind == "mlpord":
                            p = torch.sigmoid(o)
                            surv = torch.cat(
                                [torch.ones_like(p[:, :1]), torch.cumprod(p, dim=1)], dim=1
                            )
                            nxt = torch.cat(
                                [torch.cumprod(p, dim=1), torch.zeros_like(p[:, :1])], dim=1
                            )
                            prob = surv - nxt
                            out_chunks.append(
                                (prob / prob.sum(dim=1, keepdim=True).clamp_min(1e-8) @ ord_centers)
                                .cpu()
                                .numpy()
                            )
                        else:
                            out_chunks.append(o[:, 0].cpu().numpy())
                preds.append(np.concatenate(out_chunks).astype(np.float64))
                del model
                torch.cuda.empty_cache()
    raw = np.mean(preds, axis=0)[SUBMIT_MASK]
    MEMBER_Z[name] = (raw - raw.mean()) / raw.std()
    print(
        f"{name:<14}{len(seeds):>3} seeds  mean {raw.mean():+.5f}  sd {raw.std():.5f}", flush=True
    )
    del preds, raw
    gc.collect()

M_SUB = np.column_stack([MEMBER_Z[n] for n in MEMBER_ORDER])


### Combiners

In [ ]:
NL_Z = {}
for scope, family, fname in (
    ("kept", "lgbm", "v46_lgbm_residual.txt"),
    ("all", "xgb", "v47_xgb_residual.ubj"),
):
    spec = json.loads((COMBINER_DIR / f"nnls_{scope}.json").read_text())
    cols = np.array(spec["columns"])
    w = np.array(spec["weights"], dtype=np.float64)
    Z = M_SUB[:, cols]
    lz = Z @ w
    xs = np.column_stack([Z, lz])
    if family == "lgbm":
        booster = lgb.Booster(model_file=str(COMBINER_DIR / fname))
        f = booster.predict(xs)
    else:
        booster = xgb.Booster()
        booster.load_model(str(COMBINER_DIR / fname))
        f = booster.predict(xgb.DMatrix(xs))
    p = lz + f
    NL_Z[scope] = (p - p.mean()) / p.std()
    print(
        f"{scope:>5}  {len(cols)} members  {family} on residual  corr with linear {np.corrcoef(p, lz)[0, 1]:.7f}"
    )

### Write

In [ ]:
W_TOTAL = 5.368096
TARGET_MEAN = 2.3312
SIGMA = float(np.sqrt(W_TOTAL))
SE_B = 4.2e-5
RIDGE = 1.0e-4
BASE = "v43_calib"
NEW = ["v46_nl_kept", "v47_nl_all"]
NL_BETA = {"kept": 1.6276200554303766, "all": 1.6275976097615987}
NL_FILE = {"kept": "submit_v46_nl_kept", "all": "submit_v47_nl_all"}

In [ ]:
for scope in ("kept", "all"):
    z = NL_Z[scope]
    beta = NL_BETA[scope]
    lo, hi = -30.0, 30.0
    for _ in range(200):
        mid = 0.5 * (lo + hi)
        if float(np.clip(beta * z + mid, 0.0, None).mean()) < TARGET_MEAN:
            lo = mid
        else:
            hi = mid
    level = 0.5 * (lo + hi)
    pred = np.expm1(np.clip(beta * z + level, 0.0, None))
    pl.DataFrame({"user_id": USER_IDS, "predict": pred.astype(np.float32)}).write_csv(
        SUBS / f"{NL_FILE[scope]}.csv"
    )
    print(f"{NL_FILE[scope]:<22} beta {beta:.7f} level {level:.5f} zeros {(pred == 0).mean():.4%}")

### Archive

In [ ]:
V36_ALPHA, V36_COV, V36_B = 0.012, 0.009695, 1.6295100386053305
SCORES = json.loads((ARCHIVE / "scores.json").read_text())
REF_SCORES = json.loads((REFERENCE / "scores.json").read_text())

d36 = pl.read_csv(ARCHIVE / "submit_v36_fleet.csv").sort("user_id")
lg36 = np.log1p(np.clip(d36["predict"].to_numpy().astype(np.float64), 0.0, None))
lv36, bt36 = float(lg36.mean()), float(lg36.std())
b36 = (V36_COV * V36_ALPHA + V36_B) / np.sqrt(1.0 + V36_ALPHA * V36_ALPHA)
SCORES["v36_fleet"] = [
    "submit_v36_fleet",
    float(np.sqrt(W_TOTAL + bt36 * bt36 + (lv36 - TARGET_MEAN) ** 2 - 2.0 * bt36 * b36)),
]
print("v36_fleet score recovered from its probe:", SCORES["v36_fleet"][1])

STEM = {k: str(v[0]) for k, v in SCORES.items()}
LB = {k: float(v[1]) for k, v in SCORES.items()}
for k, scope in zip(NEW, ("kept", "all"), strict=True):
    STEM[k] = NL_FILE[scope]
    LB[k] = float(REF_SCORES[k][1])

NAMES, Z_ARCH, RHO, BETAS = [], [], [], []
ids = None
for n in sorted(LB, key=lambda k: LB[k]):
    stem, lb = STEM[n], LB[n]
    path = (SUBS / f"{stem}.csv") if n in NEW else (ARCHIVE / f"{stem}.csv")
    d = pl.read_csv(path).sort("user_id")
    uid = d["user_id"].to_numpy()
    ids = uid if ids is None else ids
    lg = np.log1p(np.clip(d["predict"].to_numpy().astype(np.float64), 0.0, None))
    level, beta = float(lg.mean()), float(lg.std())
    NAMES.append(n)
    Z_ARCH.append((lg - level) / beta)
    BETAS.append(beta)
    RHO.append(
        (beta * beta + W_TOTAL + (level - TARGET_MEAN) ** 2 - lb * lb) / (2.0 * beta * SIGMA)
    )
Z_ARCH = np.vstack(Z_ARCH)
RHO = np.asarray(RHO)
BETAS = np.asarray(BETAS)
NAME_IDX = {n: i for i, n in enumerate(NAMES)}
print(f"{'name':<24}{'public LB':>13}{'rho':>11}{'beta':>9}")
for i, n in enumerate(NAMES):
    print(f"{n:<24}{LB[n]:>13.7f}{RHO[i]:>11.7f}{BETAS[i]:>9.4f}")

### New directions

In [ ]:
old = [i for i, n in enumerate(NAMES) if n not in NEW]
Zo = Z_ARCH[old]
Co = (Zo @ Zo.T) / Z_ARCH.shape[1]
qo = RHO[old]
BASIS, COVS, NORMS = [], [], []
for nm in NEW:
    i = NAME_IDX[nm]
    z = Z_ARCH[i]
    c = np.linalg.solve(Co + RIDGE * np.eye(len(old)), Zo @ z / z.size)
    r = z - c @ Zo
    lb = LB[nm]
    b_obs = (W_TOTAL + BETAS[i] * BETAS[i] - lb * lb) / (2.0 * BETAS[i])
    b_span = SIGMA * float(c @ qo)
    cov_r = b_obs - b_span
    for e, ce in zip(BASIS, COVS, strict=True):
        g = float(r @ e / r.size)
        r = r - g * e
        cov_r = cov_r - g * ce
    nrm = float(np.sqrt(r @ r / r.size))
    BASIS.append(r / nrm)
    COVS.append(cov_r / nrm)
    NORMS.append(nrm)
    print(f"{nm:<14}{b_obs:>12.5f}{b_span:>15.5f}{nrm:>10.4f}{cov_r / nrm:>+13.5f}")

### Submission

In [ ]:
base_lb = LB[BASE]
b43 = float(np.sqrt(W_TOTAL - base_lb * base_lb))
DB = [c * n for c, n in zip(COVS, NORMS, strict=True)]
SHRINK = [d * d / (d * d + SE_B * SE_B) for d in DB]
ALPHA = [k * c / b43 for k, c in zip(SHRINK, COVS, strict=True)]
a2 = float(sum(a * a for a in ALPHA))
b_exp = (b43 + float(sum(a * c for a, c in zip(ALPHA, COVS, strict=True)))) / np.sqrt(1.0 + a2)
s_exp = float(np.sqrt(W_TOTAL - b_exp * b_exp))
print(f"{'direction':>10}{'Cov':>10}{'dB':>11}{'z':>7}{'shrink':>9}{'alpha':>10}")
for j, (c, d, k, a) in enumerate(zip(COVS, DB, SHRINK, ALPHA, strict=True)):
    print(f"{j + 1:>10}{c:>+10.5f}{d:>11.6f}{d / SE_B:>7.2f}{k:>9.3f}{a:>10.6f}")
print(f"\nexpected B {b_exp:.7f} -> score {s_exp:.7f}  ({base_lb - s_exp:+.7f} against {BASE})")

z_out = Z_ARCH[NAME_IDX[BASE]].copy()
for a, e in zip(ALPHA, BASIS, strict=True):
    z_out = z_out + a * e
z_out = (z_out - z_out.mean()) / z_out.std()
lo, hi = -30.0, 30.0
for _ in range(200):
    mid = 0.5 * (lo + hi)
    if float(np.clip(b_exp * z_out + mid, 0.0, None).mean()) < TARGET_MEAN:
        lo = mid
    else:
        hi = mid
level = 0.5 * (lo + hi)
pred = np.expm1(np.clip(b_exp * z_out + level, 0.0, None))
OUT_PATH = SUBS / "submit_v48_newdir.csv"
pl.DataFrame({"user_id": ids, "predict": pred.astype(np.float32)}).write_csv(OUT_PATH)
print(
    f"  beta {b_exp:.5f}  level {level:.5f}  zeros {(pred == 0).mean():.3%}  max {pred.max():.1f}"
)

### Verify

In [ ]:
chk = pl.read_csv(OUT_PATH)
assert chk.columns == ["user_id", "predict"]
assert chk.height == N_USERS
assert chk["user_id"].n_unique() == N_USERS
assert chk["user_id"].sort().to_list() == sorted(USER_IDS.tolist())
vals = chk["predict"].to_numpy()
assert np.isfinite(vals).all()
assert float(vals.min()) >= 0.0

COMPARE = {}
print(f"\n{'file':<24}{'corr vs archived':>18}{'max abs diff':>15}")
for stem in ("submit_v46_nl_kept", "submit_v47_nl_all", "submit_v48_newdir"):
    fresh = (
        pl.read_csv(SUBS / f"{stem}.csv").sort("user_id")["predict"].to_numpy().astype(np.float64)
    )
    archived = (
        pl
        .read_csv(REFERENCE / f"{stem}.csv")
        .sort("user_id")["predict"]
        .to_numpy()
        .astype(np.float64)
    )
    c = float(
        np.corrcoef(np.log1p(np.clip(fresh, 0, None)), np.log1p(np.clip(archived, 0, None)))[0, 1]
    )
    d = float(np.abs(fresh - archived).max())
    COMPARE[stem] = {"corr": c, "max_abs_diff": d}
    print(f"{stem:<24}{c:>18.9f}{d:>15.4f}")

### Log

In [ ]:
PUBLIC_LB = 1.6460816563
with mlflow.start_run(run_name="prod-infer"):
    mlflow.set_tags({"pipeline": "prod", "stage": "inference", "submission_file": OUT_PATH.name})
    mlflow.log_params({
        "members": ",".join(MEMBER_ORDER),
        "n_members": len(MEMBER_ORDER),
        "submit_anchor": SUBMIT,
        "base": BASE,
        "archive_vectors": len(NAMES),
        "ridge": RIDGE,
        "se_b": SE_B,
        "architecture": (
            "34 member ensemble, non-negative linear blend, LightGBM and XGBoost residual "
            "combiners, then v43 plus the two directions v46 and v47 add outside the span of "
            "the 63 scored predecessors, each shrunk by its own significance"
        ),
    })
    mlflow.log_metrics({
        "b43": b43,
        "b_expected": b_exp,
        "score_expected": s_exp,
        "alpha_1": ALPHA[0],
        "alpha_2": ALPHA[1],
        "cov_1": COVS[0],
        "cov_2": COVS[1],
        "public_lb_rmsle": PUBLIC_LB,
    })
    mlflow.log_metrics({f"corr_vs_archived_{k}": v["corr"] for k, v in COMPARE.items()})
    mlflow.log_artifact(str(OUT_PATH), "submissions")
    mlflow.log_dict(COMPARE, "compare_vs_archived.json")